In [0]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, 
    DoubleType, IntegerType, ArrayType
)

In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
input_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("order_count", IntegerType(), True),
    StructField("total_spend", DoubleType(), True)
])

In [0]:
data = [
    # Gold Segment
    ("C001", "Gold", 12, 1450.50), ("C002", "Gold", 25, 3100.00),
    ("C003", "Gold", 8, 920.75),   ("C004", "Gold", 18, 2200.10),
    ("C005", "Gold", 30, 4150.00), ("C006", "Gold", 15, 1890.30),
    ("C007", "Gold", 5, 600.00),   ("C008", "Gold", 22, 2800.50),
    ("C009", "Gold", 19, 2100.00), ("C010", "Gold", 14, 1650.80),
    ("C011", "Gold", 27, 3900.20), ("C012", "Gold", 9, 1100.00),

    # Silver Segment
    ("C013", "Silver", 6, 450.00),  ("C014", "Silver", 10, 890.50),
    ("C015", "Silver", 4, 310.20),  ("C016", "Silver", 8, 670.00),
    ("C017", "Silver", 11, 950.40), ("C018", "Silver", 3, 210.00),
    ("C019", "Silver", 7, 580.90),  ("C020", "Silver", 12, 1020.00),
    ("C021", "Silver", 5, 410.00),  ("C022", "Silver", 9, 780.30),
    ("C023", "Silver", 2, 150.00),  ("C024", "Silver", 6, 500.00),

    # Bronze / New Segment
    ("C025", "Bronze", 1, 45.00),   ("C026", "Bronze", 3, 120.50),
    ("C027", "Bronze", 2, 85.00),   ("C028", "Bronze", 1, 30.00),
    ("C029", "Bronze", 4, 190.00),  ("C030", "Bronze", 2, 95.50),
    ("C031", "Bronze", 5, 230.00),  ("C032", "Bronze", 1, 50.00),
    ("C033", "Bronze", 3, 140.00),  ("C034", "Bronze", 2, 110.00),
    ("C035", "Bronze", 4, 210.00)
]

In [0]:
df = spark.createDataFrame(data, schema=input_schema)

In [0]:
def process_segment(pdf: pd.DataFrame) -> pd.DataFrame:
    """
    Executes per group (segment). 
    Calculates average order value and standardizes (z-score) spend per group.
    """
    pdf["avg_spend_per_order"] = (pdf["total_spend"] / pdf["order_count"]).round(2)
    
    # Calculate Z-score per segment
    mean_spend = pdf["total_spend"].mean()
    std_spend = pdf["total_spend"].std(ddof=0)
    
    pdf["segment_spend_zscore"] = (
        (pdf["total_spend"] - mean_spend) / (std_spend if std_spend > 0 else 1)
    ).round(2)
    
    # Filter output columns matching output_schema
    return pdf[[
        "segment", 
        "customer_id", 
        "total_spend", 
        "avg_spend_per_order", 
        "segment_spend_zscore"
    ]]

In [0]:
output_schema = StructType([
    StructField("segment", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("total_spend", DoubleType(), True),
    StructField("avg_spend_per_order", DoubleType(), True),
    StructField("segment_spend_zscore", DoubleType(), True)
])

In [0]:
result_df = df.groupby("segment").applyInPandas(process_segment, schema=output_schema)

In [0]:
result_df.sort("segment", "total_spend", ascending=False).show(10, truncate=False)